# Project: Precision Pharmacology Pipeline

This file aims to extract the "signatures" from the main 20GB file, merge the metadata, and save the "significant" rows to a CSV.

## Phase 1
### Step 1: Initialize and Filter Metadata

We need to load our metadata files and generate lists of IDs for the extraction:

1. **Row IDs (rid):** The 978 Landmark Genes.
2. **Column IDs (cid):** The specific drug experiments (Signatures) done on MCF7 cells at a 10µM dose.
3. **Compound Data:** The structural chemistry data (SMILES/PubChem) for all perturbations.

In [3]:
import os
import h5py
import pandas as pd
from cmapPy.pandasGEXpress.parse import parse

# --- CONFIGURATION ---
GCTX_FILE = 'GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx'
GENE_METADATA = 'GSE92742_Broad_LINCS_gene_info.txt.gz'
SIG_METADATA = 'GSE92742_Broad_LINCS_sig_info.txt.gz'
PERT_INFO_FILE = 'GSE92742_Broad_LINCS_pert_info.txt'

# 1. VERIFY FILES
for f in [GCTX_FILE, GENE_METADATA, SIG_METADATA, PERT_INFO_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing file: {f}. Please ensure it is in the current directory.")

# 2. IDENTIFY LANDMARK GENES (Rows)
df_gene = pd.read_csv(GENE_METADATA, sep='\t', compression='gzip')
landmark_rids = df_gene[df_gene['pr_is_lm'] == 1]['pr_gene_id'].astype(str).tolist()

# 3. IDENTIFY TARGET SIGNATURES (Columns)
df_sig = pd.read_csv(SIG_METADATA, sep='\t', compression='gzip')
mcf7_10um_sigs = df_sig[
    (df_sig['cell_id'] == 'MCF7') & 
    (df_sig['pert_dose'] == 10) &
    (df_sig['pert_dose_unit'].isin(['µm', 'µM']))
]['sig_id'].tolist()

# 4. LOAD COMPOUND STRUCTURE METADATA
df_pert_info = pd.read_csv(PERT_INFO_FILE, sep='\t')

C:\Users\Jawad\AppData\Local\Temp\ipykernel_13644\2549284560.py:22: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sig = pd.read_csv(SIG_METADATA, sep='\t', compression='gzip')


In [3]:
print(f"df_gene shape: {df_gene.shape}")
print(f"\ndf_sig shape: {df_sig.shape}")

df_gene shape: (12328, 5)

df_sig shape: (473647, 12)


In [4]:
print(f"df_gene info: \n{df_gene.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12328 entries, 0 to 12327
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   pr_gene_id      12328 non-null  int64 
 1   pr_gene_symbol  12328 non-null  object
 2   pr_gene_title   12328 non-null  object
 3   pr_is_lm        12328 non-null  int64 
 4   pr_is_bing      12328 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 481.7+ KB
df_gene info: 
None


In [5]:
print(f"df_sig info: \n{df_sig.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 473647 entries, 0 to 473646
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   sig_id          473647 non-null  object
 1   pert_id         473647 non-null  object
 2   pert_iname      473647 non-null  object
 3   pert_type       473647 non-null  object
 4   cell_id         473647 non-null  object
 5   pert_dose       473647 non-null  object
 6   pert_dose_unit  473647 non-null  object
 7   pert_idose      473647 non-null  object
 8   pert_time       473647 non-null  int64 
 9   pert_time_unit  473647 non-null  object
 10  pert_itime      473647 non-null  object
 11  distil_id       473647 non-null  object
dtypes: int64(1), object(11)
memory usage: 43.4+ MB
df_sig info: 
None


In [6]:
df_gene.head()

,pr_gene_id,pr_gene_symbol,pr_gene_title,pr_is_lm,pr_is_bing
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
2,2978,GUCA1A,guanylate cyclase activator 1A,0,0
3,2049,EPHB3,EPH receptor B3,0,1
4,2101,ESRRA,estrogen related receptor alpha,0,1


In [7]:
print(f"df_sig head: \n{df_sig.head()}")
# sig_id ending with :A05 (specific to a physical well on a plate)

df_sig head: 
                                  sig_id        pert_id     pert_iname  \
0                    AML001_CD34_24H:A05           DMSO           DMSO   
1                    AML001_CD34_24H:A06           DMSO           DMSO   
2                    AML001_CD34_24H:B05           DMSO           DMSO   
3                    AML001_CD34_24H:B06           DMSO           DMSO   
4  AML001_CD34_24H:BRD-A03772856:0.37037  BRD-A03772856  BRD-A03772856   

     pert_type cell_id pert_dose pert_dose_unit pert_idose  pert_time  \
0  ctl_vehicle    CD34       0.1              %      0.1 %         24   
1  ctl_vehicle    CD34       0.1              %      0.1 %         24   
2  ctl_vehicle    CD34       0.1              %      0.1 %         24   
3  ctl_vehicle    CD34       0.1              %      0.1 %         24   
4       trt_cp    CD34   0.37037             µM     500 nM         24   

  pert_time_unit pert_itime                                          distil_id  
0              h     

In [4]:
print(f"\ndf_per_info shape: {df_pert_info.shape}")


df_per_info shape: (51383, 8)


In [5]:
print(f"df_pert_info info: \n{df_pert_info.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51383 entries, 0 to 51382
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   pert_id           51383 non-null  object
 1   pert_iname        51383 non-null  object
 2   pert_type         51383 non-null  object
 3   is_touchstone     51383 non-null  int64 
 4   inchi_key_prefix  51383 non-null  object
 5   inchi_key         51383 non-null  object
 6   canonical_smiles  51383 non-null  object
 7   pubchem_cid       51383 non-null  object
dtypes: int64(1), object(7)
memory usage: 3.1+ MB
df_pert_info info: 
None


In [8]:
df_pert_info.iloc[1000:1005]

,pert_id,pert_iname,pert_type,is_touchstone,inchi_key_prefix,inchi_key,canonical_smiles,pubchem_cid
1000,BRD-A61858259,CAY-10415,trt_cp,1,IRNJSRAGRIZIHD,IRNJSRAGRIZIHD-UHFFFAOYSA-N,CCc1ccc(nc1)C(=O)COc1ccc(CC2SC(=O)NC2=O)cc1,10429242
1001,BRD-A61866602,5-methoxy-alpha-methyltryptamine,trt_cp,0,OGNJZVNNKBZFRM,OGNJZVNNKBZFRM-UHFFFAOYSA-N,COc1ccc2[nH]cc(CC(C)N)c2c1,36906
1002,BRD-A61899133,catechin,trt_cp,0,PFTAWBLQPZVEMU,PFTAWBLQPZVEMU-UHFFFAOYSA-N,OC1Cc2c(O)cc(O)cc2OC1c3ccc(O)c(O)c3,-666
1003,BRD-A62021152,WAY-161503,trt_cp,1,PHGWDAICBXUJDU,PHGWDAICBXUJDU-UHFFFAOYSA-N,Clc1cc2NC(=O)C3CNCCN3c2cc1Cl,-666
1004,BRD-A62025033,temsirolimus,trt_cp,1,CBPNZQVSJQDFBE,CBPNZQVSJQDFBE-WEDGMEQVSA-N,COC1CC(CC(C)C2CC(=O)C(C)\C=C(C)\C(O)C(OC)C(=O)...,6440033


### Step 2: Extract the Data (The "Surgery")

In [8]:
!pip install cmapPy

In [9]:
# 1. VALIDATE IDS AGAINST GCTX FILE HEADERS
with h5py.File(GCTX_FILE, 'r') as f:
    actual_rids = [x.decode('utf-8') for x in f['0/META/ROW/id'][:]]
    actual_cids = [x.decode('utf-8') for x in f['0/META/COL/id'][:]]

valid_rids = list(set(landmark_rids).intersection(actual_rids))
valid_cids = list(set(mcf7_10um_sigs).intersection(actual_cids))

print(f"Targeting {len(valid_rids)} Landmark Genes across {len(valid_cids)} MCF7 experiments.")

# 2. THE SURGERY (Extraction)
print("Starting extraction from GCTX file...")
gctoo_data = parse(GCTX_FILE, rid=valid_rids, cid=valid_cids)

df_lincs_final = gctoo_data.data_df

print(f"✅ Extraction Successful! Matrix Shape: {df_lincs_final.shape}")

Targeting 978 Landmark Genes across 14861 MCF7 experiments.
Starting extraction from GCTX file...


c:\Users\Jawad\anaconda3\Lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
c:\Users\Jawad\anaconda3\Lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


✅ Extraction Successful! Matrix Shape: (978, 14861)


In [10]:
print(f"gctoo_data: {gctoo_data}")

gctoo_data: b'GCTX1.0'
src: GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx
data_df: [978 rows x 14861 columns]
row_metadata_df: [978 rows x 0 columns]
col_metadata_df: [14861 rows x 0 columns]


In [11]:
print(f"gctoo_data data_df shape: {gctoo_data.data_df.shape}")
print(f"df_lincs_final shape: {df_lincs_final.shape}")

gctoo_data data_df shape: (978, 14861)
df_lincs_final shape: (978, 14861)


In [12]:
print(f"df_lincs_final null: \n{df_lincs_final.isna().sum()}")

df_lincs_final null: 
cid
CPC004_MCF7_6H:BRD-A66861218-001-03-5:10      0
CPC004_MCF7_6H:BRD-A29322418-237-03-2:10      0
CPC004_MCF7_6H:BRD-K08109516-001-02-9:10      0
CPC004_MCF7_6H:BRD-K88741031-001-01-0:10      0
CPC004_MCF7_6H:BRD-K16604360-001-01-8:10      0
                                             ..
PCLB003_MCF7_24H:BRD-K78431006-001-05-2:10    0
PCLB003_MCF7_24H:BRD-K63828191-003-23-0:10    0
PCLB003_MCF7_24H:BRD-K93754473-001-17-7:10    0
PCLB003_MCF7_24H:BRD-A52530684-003-01-7:10    0
PCLB003_MCF7_24H:BRD-A75409952-001-01-6:10    0
Length: 14861, dtype: int64


In [13]:
print(f"df_lincs_final duplicates: {df_lincs_final.duplicated().sum()}")

df_lincs_final duplicates: 0


In [14]:
print(f"df_lincs_final info: {df_lincs_final.info()}")

<class 'pandas.core.frame.DataFrame'>
Index: 978 entries, 5720 to 874
Columns: 14861 entries, CPC004_MCF7_6H:BRD-A66861218-001-03-5:10 to PCLB003_MCF7_24H:BRD-A75409952-001-01-6:10
dtypes: float32(14861)
memory usage: 55.5+ MB
df_lincs_final info: None


In [15]:
print(f"df_lincs_final head: \n{df_lincs_final.iloc[:, :3].head()}")

df_lincs_final head: 
cid   CPC004_MCF7_6H:BRD-A66861218-001-03-5:10  \
rid                                              
5720                                 -0.039385   
466                                   0.138223   
6009                                 -0.313118   
2309                                 -0.115185   
387                                  -0.903532   

cid   CPC004_MCF7_6H:BRD-A29322418-237-03-2:10  \
rid                                              
5720                                  0.062316   
466                                  -0.211403   
6009                                  0.690583   
2309                                  0.075742   
387                                   0.893083   

cid   CPC004_MCF7_6H:BRD-K08109516-001-02-9:10  
rid                                             
5720                                  0.625000  
466                                  -0.136795  
6009                                 -0.524686  
2309                           

### Step 3: Melt for SQL Compatibility

In [16]:
# 1. Convert from Matrix (Wide) to Long format
# SQL databases prefer "Long" tables over "Wide" ones. We need to turn this matrix into a list of individual observations.
print("Melting Matrix to Long Format...")
df_long = df_lincs_final.stack().reset_index()
df_long.columns = ['gene_id', 'sig_id', 'z_score']

# Ensure IDs are strings for clean merging
df_long['gene_id'] = df_long['gene_id'].astype(str)
df_gene['pr_gene_id'] = df_gene['pr_gene_id'].astype(str)

# ==========================================
# NEW RELATIONAL ARCHITECTURE PREPARATION
# ==========================================

# 2. Attach Gene Symbols to the Long Data
print("Attaching Gene Symbols...")
df_long = pd.merge(
    df_long, 
    df_gene[['pr_gene_id', 'pr_gene_symbol']], 
    left_on='gene_id', 
    right_on='pr_gene_id', 
    how='left'
).drop(columns=['pr_gene_id', 'gene_id'])

# 3. Filter for Treatment Compounds ONLY using the Metadata
print("Filtering for Treatment Compounds (trt_cp)...")
valid_sigs = df_sig[df_sig['pert_type'] == 'trt_cp']['sig_id']
df_long_filtered = df_long[df_long['sig_id'].isin(valid_sigs)].copy()

# 4. Attach the 'pert_iname' to the expression data (This acts as our Foreign Key)
print("Creating Foreign Keys...")
df_expression = pd.merge(
    df_long_filtered,
    df_sig[['sig_id', 'pert_iname']],
    on='sig_id',
    how='left'
)

# 5. Build the Metadata Table (The "Left Side" of the JOIN)
# Combine local structural chemistry with live biological mechanisms

print("Using pre-loaded local compound structures (pert_info)...")

print("Downloading Official Drug Repurposing Hub Metadata (MoA and Targets)...")
# The official Broad Institute S3 bucket URL for the Repurposing Hub
repurposing_url = "https://s3.amazonaws.com/data.clue.io/repurposing/downloads/repurposing_drugs_20200324.txt"
# We use comment='!' because the Broad Institute sometimes puts text notes at the top of their files
df_repurposing = pd.read_csv(repurposing_url, sep='\t', comment='!', low_memory=False)

print("Merging Structural and Biological Metadata...")
# Merge the two metadata sources on the drug name (pert_iname)
df_combined_meta = pd.merge(
    df_pert_info[['pert_iname', 'canonical_smiles', 'pubchem_cid']], 
    df_repurposing[['pert_iname', 'moa', 'target']], 
    on='pert_iname', 
    how='left'
)

# Get the unique list of drugs we successfully filtered in Step 4
unique_drugs = df_expression['pert_iname'].unique()

# Filter our combined metadata to only include the specific drugs in our MCF7 project
df_metadata = df_combined_meta[df_combined_meta['pert_iname'].isin(unique_drugs)].copy()

# Drop any accidental duplicates to ensure strict Primary Key integrity for our SQL Database
df_metadata = df_metadata.drop_duplicates(subset=['pert_iname'])

print("\nRelational Split Complete:")
print(f"Expression Table Rows: {len(df_expression)}")
print(f"Metadata Table Rows (Unique Drugs): {len(df_metadata)}")

Melting Matrix to Long Format...
Attaching Gene Symbols...
Filtering for Treatment Compounds (trt_cp)...
Creating Foreign Keys...
Using pre-loaded local compound structures (pert_info)...
Merging Structural and Biological Metadata...

Relational Split Complete:
Expression Table Rows: 14534058
Metadata Table Rows (Unique Drugs): 5321


### Important: The "Size" Warning for the next Step

14.5 million rows is a lot for a standard SQL database on a personal laptop. Before we move to the SQL phase, we can save a filtered version of the data.

In drug repurposing, we usually only care about "significant" hits. A Z-score of 0.06 (like row index 1) means the drug did absolutely nothing to that gene.

In [17]:
### Important: The "Size" Warning and Database Export Strategy ###

# 1. Filter the Expression Data for 'Significant' hits only (Z-score > 2 or < -2). 
# This reduces the row count drastically, optimizing our SQL database.
print("Filtering for statistically significant Z-scores...")
df_expression_sig = df_expression[df_expression['z_score'].abs() >= 2.0].copy()

# Add a boolean flag column just in case we need it for easier querying later
df_expression_sig['is_significant'] = 1 

print(f"Significant expression rows remaining: {len(df_expression_sig)}")

# 2. Save BOTH tables to CSV to secure our progress.
# These will be the foundation of our Relational SQL Database in Phase 2.
print("\nExporting to CSV...")
df_metadata.to_csv('lincs_compound_metadata.csv', index=False)
print("✅ Saved 'lincs_compound_metadata.csv'")

# We keep only the necessary columns to save disk space
columns_to_keep = ['pert_iname', 'sig_id', 'pr_gene_symbol', 'z_score', 'is_significant']
df_expression_sig[columns_to_keep].to_csv('lincs_genetic_signatures.csv', index=False)
print("✅ Saved 'lincs_genetic_signatures.csv'")

print("\n🚀 Phase 1 officially complete. Ready for Relational Database ingestion.")

Filtering for statistically significant Z-scores...
Significant expression rows remaining: 1049972

Exporting to CSV...
✅ Saved 'lincs_compound_metadata.csv'
✅ Saved 'lincs_genetic_signatures.csv'

🚀 Phase 1 officially complete. Ready for Relational Database ingestion.
